# Tracking experiments two ways — SageMaker Experiments and MLflow

One question, two trackers, same account.

The question is the one from Demo 7: *we scored 0.92 and were delighted —
was the model good, or was the question easy?*

| Part | What it shows |
|---|---|
| 1 | **SageMaker Experiments** — the five runs already in the account |
| 2 | **MLflow** — the same arc, logged to SageMaker's managed MLflow |
| 3 | Logging a run **live**, so you see `mlflow.log_metric` happen |
| 4 | The two side by side, and which to pick |

**Cost: $0.** Parts 1, 2 and 4 only read. Part 3 trains one small model on this
notebook's own instance. Nothing hourly is created.


## 0. Setup

`mlflow` and `sagemaker-mlflow` are not in the stock Studio image. The second one
is the plugin that lets MLflow authenticate to SageMaker with your IAM role —
without it, setting an ARN as the tracking URI fails.


In [ ]:
%pip install -q mlflow sagemaker-mlflow


If that printed a restart notice, restart the kernel before continuing.


In [ ]:
import boto3
import pandas as pd

# Import submodules EXPLICITLY. `import sagemaker` does not reliably bind
# `sagemaker.image_uris` / `sagemaker.get_execution_role` -- it depends on what
# that version's __init__.py re-exports, which differs between the Studio image
# and a laptop install. That difference killed a live demo once.
from sagemaker.session import get_execution_role

sess    = boto3.Session()
region  = sess.region_name or 'us-east-1'
account = sess.client('sts').get_caller_identity()['Account']
sm      = sess.client('sagemaker')

buckets = [b['Name'] for b in sess.client('s3').list_buckets()['Buckets']
           if 'smdemo' in b['Name']]
BUCKET  = buckets[0] if buckets else f'roi-smdemo-{account}-{region}'

print(f'region  {region}')
print(f'account {account}')
print(f'bucket  {BUCKET}')


## 1. SageMaker Experiments — what is already there

Five runs: the baseline training job plus the four HPO trials. They were logged
by the training jobs themselves, not by this notebook.


In [ ]:
EXPERIMENT = 'roi-smdemo-gas-experiment'

rows = []
for c in sm.list_trial_components(ExperimentName=EXPERIMENT)['TrialComponentSummaries']:
    d = sm.describe_trial_component(TrialComponentName=c['TrialComponentName'])
    params  = {k: (v.get('StringValue') or v.get('NumberValue'))
               for k, v in d.get('Parameters', {}).items()}
    metrics = {m['MetricName']: m.get('Last') for m in d.get('Metrics', [])}
    rows.append({
        'run': d['TrialComponentName'].replace(EXPERIMENT + '-', ''),
        'learning_rate': params.get('learning-rate'),
        'max_depth': params.get('max-depth'),
        'validation_auc': metrics.get('validation_auc'),
        'validation_accuracy': metrics.get('validation_accuracy'),
    })

sm_runs = pd.DataFrame(rows).sort_values('validation_auc', ascending=False)
sm_runs.reset_index(drop=True)


Baseline 0.9235, best HPO trial 0.9365. **Four tuning trials moved the objective
by about one and a third points.** Hold that thought.

> `validation_accuracy` is `NaN` for the HPO rows and that is correct, not broken —
> a tuning job logs only the objective metric it was told to optimise
> (`validation:auc`). The baseline job logged both.

In the console this is **SageMaker AI → Experiments → `roi-smdemo-gas-experiment`**,
select all five → **Compare**.


## 2. MLflow — the same question, a different tracker

This account has a managed **MLflow App**. Two things worth knowing before you
put one in a budget:

| | Billing |
|---|---|
| MLflow **App** (`mlflow-app/...`) | serverless, no start/stop, **$0 at rest** |
| MLflow **Tracking Server** (`mlflow-tracking-server/...`) | **$0.60/hr** Small, from Created until Deleted |

Both are addressed by ARN, so the ARN is the only thing that tells them apart.
This notebook uses the App, which is why it costs nothing.


In [ ]:
import mlflow

MLFLOW_ARN = f'arn:aws:sagemaker:{region}:{account}:mlflow-app/app-D7XHWR77EVWN'

mlflow.set_tracking_uri(MLFLOW_ARN)
client = mlflow.MlflowClient()

for e in client.search_experiments():
    print(f'{e.experiment_id:>3}  {e.name}')


In [ ]:
ABLATION = 'gas-demand-the-honest-metric'
exp = client.get_experiment_by_name(ABLATION)

runs = client.search_runs([exp.experiment_id])
mlf = pd.DataFrame([{
    'run': r.info.run_name,
    'features': r.data.params.get('n_features'),
    'label_definition': r.data.params.get('label_definition'),
    'validation_auc': r.data.metrics.get('validation_auc'),
    'validation_accuracy': r.data.metrics.get('validation_accuracy'),
} for r in runs]).sort_values('validation_auc', ascending=False)

mlf.reset_index(drop=True)


### The arc

Read it top to bottom. Everything except the last row uses the **same label**.

- **all-features 0.92** — the number everyone is happy with
- **calendar-only 0.90** — month and season alone, *no weather reading at all*
- **weather-only 0.83** — what the station actually measured
- **deseasonalised 0.52** — same features, harder question, coin flip

The model was never the variable. The *question* was.


In [ ]:
# Why: feature importance was logged alongside the metrics.
best = [r for r in runs if r.info.run_name == 'all-features'][0]

imp = {k.replace('importance__', ''): v
       for k, v in best.data.metrics.items() if k.startswith('importance__')}
pd.Series(imp).sort_values(ascending=False).to_frame('importance')


`season_Winter` alone is about **70%** of the model. It learned *'winter is cold
and people heat their homes'* — true, and it teaches nothing.


## 3. Log a run live

Everything above was read back. This writes. Small on purpose so it finishes while
you are talking — 15,000 rows and 40 trees, not the full 105,000 and 150.


In [ ]:
import io
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

s3 = sess.client('s3')
def read_csv(key):
    return pd.read_csv(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()))

train = read_csv('ml/gas/train/train.csv').sample(n=15000, random_state=42)
valid = read_csv('ml/gas/validation/validation.csv')

LABEL = 'high_demand'
cols  = [c for c in train.columns if c != LABEL]
print(f'{len(train):,} train rows x {len(cols)} features')


In [ ]:
mlflow.set_experiment(ABLATION)

with mlflow.start_run(run_name='live-in-class') as run:
    hp = {'n_estimators': 40, 'max_depth': 3, 'learning_rate': 0.1, 'random_state': 42}
    mlflow.log_params(hp)
    mlflow.log_param('n_train_rows', len(train))
    mlflow.set_tag('note', 'trained live during the session')

    model = GradientBoostingClassifier(**hp).fit(train[cols], train[LABEL])
    auc = roc_auc_score(valid[LABEL], model.predict_proba(valid[cols])[:, 1])

    mlflow.log_metric('validation_auc', auc)
    print(f'validation AUC {auc:.4f}')
    print(f'run id {run.info.run_id}')


Three lines of logging — `log_params`, `log_metric`, `set_tag` — and the run is
in the tracker next to the others. That is the whole MLflow API that matters.

Re-run the query from Part 2 and `live-in-class` is there.


## 4. So which one?

| | SageMaker Experiments | SageMaker managed MLflow |
|---|---|---|
| Cost | free | App free · Tracking Server $0.60/hr |
| Logging | automatic from training jobs | you call `log_metric` |
| Works outside SageMaker | no | yes — it is real MLflow |
| Portable off AWS | no | yes, same API everywhere |
| Model registry | SageMaker Model Registry | auto-registers into the same registry |

**The honest answer:** if everything you train is a SageMaker training job,
Experiments gives you this for free with no code. If your team already lives in
MLflow, or trains anywhere other than SageMaker, use MLflow — and know which of
the two resource types you created, because one is free and one is $432/month if
you forget it.

Neither one would have caught the problem in Part 2. **A tracker records what you
measured. It cannot tell you the question was too easy.** That part is still the
job.


---

### Open the MLflow UI

**Studio → Applications → MLflow**, or generate a signed URL:

```python
print(sm.create_presigned_mlflow_app_url(Arn=MLFLOW_ARN, ExpiresInSeconds=300)['AuthorizedUrl'])
```

### Rebuild the ablation runs

```bash
python 05-experiments-tuning/mlflow_log_ablation.py
```
